## EDA для набора данных job_postings_with_salaries

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_data

df = load_data()
df_original = df.copy()

Сначала проверю, что представляет собой строка датасета: какие поля есть в записи о вакансии и что в них лежит

In [ ]:
df.head()

In [ ]:
df.tail()

Записи довольно объёмные: одна запись содержит 27 колонок, в первых и последних строках некоторые колонки имеют значительное число пропусков, эти колонки необходимо отдельно проверить, случанойсть ли это или же паттерн самих данных.

Данные о зарплате дублируются: **salary_min** / **salary_max** — исходные значения в своей валюте и за свой период, **salary_min_annual** / **salary_max_annual** — те же суммы за год, а **salary_text** — их текстовое описание. Значит, для анализа нужно выбрать одно представление и унифицировать данные, часть информации лучше откинуть, чтобы проще было работать.

Дальше проверяю структуру самого датасета.

In [ ]:
df.info()

Из 27 признаков числовые только четыре, и все связанные с зарплатой: **salary_min**, **salary_max**, **salary_min_annual**, **salary_max_annual**. Остальные 23 — строковые (категориальные и текстовые).

Для категориальных считаем частоты, кардинальность и пропуски, для числовых — распределение и выбросы. Даты **posted_at** и **first_seen_at** лежат строками, для анализа по времени их придётся преобразовать в `datetime`.

In [ ]:
print(df.duplicated().sum())
print("-------------------")
print(df["job_id"].duplicated().sum())
print("-------------------")
print(df.isna().sum())

Полных дубликатов нет, **job_id** уникален — это 9173 независимые записи.

Пропусков много: например, **seniority**, **region**, **city**. **seniority** не указан, возможно из-за что указан в столбце **description** или же на сайте самой компаннии, либо же должность будет обсуждаться непосредственно на собеседовании, в любом случае с пропусками нужно что-то делать, либо удалять, либо заменять.         

Ниже — визуализация пропусков, чтобы увидеть картину целиком.

In [ ]:
missing_counts = df.isna().sum().sort_values(ascending=True)
missing_counts = missing_counts[missing_counts > 0]

sns.barplot(x=missing_counts.values, y=missing_counts.index)

plt.xlabel("Количество пропусков")
plt.ylabel("Признаки")
plt.title("Количество пропусков по признакам")
plt.tight_layout()
plt.show()

Больше тысячи пропусков у восьми признаков: **seniority** (8543), **region** (6965), **city** (6787), **language** (5679), **workplace_type** (5147), **department** (3755), **employment_type** (2516), **country_code** (1122).

Зарплата указана в разных валютах (USD, EUR, CAD, GBP, AUD, PLN, NZD, CHF) и за разные периоды (год, месяц, неделя, день, час). Период обычно связан с занятостью: почасовые и частичные позиции дают ставку за час, постоянные — годовую сумму.

Необходимо изучить распределение по валютам и периодам и принять решение о том, к какому значению приводить все данные 

In [ ]:
currency_counts = df["salary_currency"].value_counts()
period_counts = df["salary_period"].value_counts()

print(currency_counts)
print(period_counts)

По валютам доминирует **USD** — 7796 вакансий, дальше **EUR** (527) и **CAD** (499), остальные валюты единичны.

По периоду преобладает год: **year** — 6639 вакансий, но и **hour** — 2243, то есть почти четверть вакансий. Периоды **month** (274), **week** (14) и **day** (3) редки.

Поэтому нормализую к **USD** и к году — самые частые значения, значит меньше строк придётся пересчитывать.

In [ ]:
currency_df = currency_counts.rename('count').reset_index()
currency_df.columns = ['currency', 'count']

sns.barplot(data=currency_df, x='currency', y='count')

plt.title("Количество вакансий по валютам")
plt.xlabel("Тип валюты")
plt.ylabel("Количество вакансий")
plt.tight_layout()
plt.show()

In [ ]:
period_df = period_counts.rename('count').reset_index()
period_df.columns = ['period', 'count']

sns.barplot(data= period_df, x='period', y='count')

plt.title("Период, за который указана заработная плата")
plt.xlabel("Указанный период")
plt.ylabel("Количество вакансий")
plt.tight_layout()
plt.show()

Приводим зарплаты к одному виду — валюта **USD** и период «год».

Период пересчитываем через стандартное число рабочих единиц в году: час × 2080, день × 260, неделя × 52, месяц × 12, год × 1. Стандартное время по меркам американского рынка труда

In [ ]:
df = df_original.copy()

dict_period_with_coef = {
    "day": 260,
    "week":52,
    "month":12,
    "hour": 2080,
    "year": 1
}

coef = df["salary_period"].map(dict_period_with_coef)
df["salary_max"] *= coef
df["salary_min"] *= coef

df['salary_period'] = df["salary_period"].replace(['day', 'week', 'hour', 'month'], 'year')


Теперь валюта: переводим суммы в **USD** по курсам Национального банка Республики Беларусь.

Коэффициенты заданы как «сколько валюты в 1 USD» (EUR 0.89, CAD 1.42, GBP 0.75, AUD 1.45, PLN 3.9, NZD 1.78, CHF 0.83), поэтому сумма делится на коэффициент: EUR 59 175 → ≈66 489 USD.

Курс зафиксирован на одну дату, так что пересчёт приблизительный, но для сравнения порядков этого достаточно.

In [ ]:
dict_currency_with_coef = {
    "USD": 1.0,
    "EUR": 0.89,
    "CAD": 1.42,
    "GBP": 0.75,
    "AUD": 1.45,
    "PLN": 3.9,
    "NZD": 1.78,
    "CHF": 0.83
}

coef_currency = df["salary_currency"].map(dict_currency_with_coef)


df["salary_max"] = (df["salary_max"] / coef_currency).round(2)
df["salary_min"] = (df["salary_min"] / coef_currency).round(2)

df['salary_currency'] = df["salary_currency"].replace(['EUR', 'CAD', 'GBP', 'AUD', 'PLN', 'NZD', 'CHF'], 'USD')

In [ ]:
df.describe().round(2)

Курсы взяты на 15.10.2026 по данным НБ Республики Беларусь. Это статические значения, а не исторические: при обновлении данных их стоит перепроверять

После нормализации **salary_currency** и **salary_period** стали константами (USD, year) и для анализа больше не нужны. А два оставшихся признака стоит посмотреть:

- **salary_text** — исходное описание зарплаты, дублирует числовые колонки, но удобен как проверка корректности пересчёта;
- **salary_source** — откуда взята зарплата (поле ATS или текст описания), потенциально полезный категориальный признак.

Проверяю 20 случайных записей и сопоставляю **salary_text** с посчитанными **salary_min** / **salary_max**: расхождения сразу показывают проблемные строки.

In [ ]:
salary_cols = [
    "salary_min",
    "salary_max",
    "salary_currency",
    "salary_period",
    "salary_text",
    "salary_source",
    "salary_min_annual",
    "salary_max_annual"
]

df[salary_cols].sample(
    n = 20,
    random_state=42
)

In [ ]:
print(df['salary_currency'].value_counts())
print("-------------------------")
print(df['salary_period'].value_counts())
print("-------------------------")
print(df['salary_source'].value_counts())

Итог по этим признакам:

- **salary_text** — то же описание зарплаты, что уже посчитано в **salary_min** / **salary_max**. Для анализа не нужен, но полезен как контроль: по нему видно, где пересчёт сработал неверно.
- **salary_currency** и **salary_period** стали константами и больше не участвуют.
- **salary_source** — полезный признак: показывает, откуда взялась зарплата, и косвенно характеризует компанию и способ сбора данных. Разбираем его дальше вместе с другими категориальными признаками.

## Взаимосвязь категориальных признаков и заработной платы

Сравниваю категориальные признаки с **salary_min** и **salary_max** — уже приведёнными к USD и году. В таблицах: **n** — число вакансий, **min_median** / **max_median** — медианы, **median_vs_all** — отношение медианы категории к общей медиане.

Зарплаты сильно смещены вправо (skew ≈ 3.8), поэтому смотрим медиану и квартили, а не среднее. В расчёт попадают только категории с **n ≥ 30**, а пропуски выделены как «(нет данных)». На boxplot'ах выбросы скрыты — им посвящён следующий раздел.

Сначала общая картина: сколько уникальных значений и пропусков в каждом признаке. От этого зависит, что вообще имеет смысл сравнивать по группам.

In [ ]:
overview = pd.DataFrame({
    "уникальных значений": df.nunique(dropna=True),
    "пропусков": df.isna().sum(),
    "доля пропусков, %": (df.isna().mean() * 100).round(1),
    "тип": df.dtypes.astype(str),
}).sort_values("уникальных значений", ascending=False)

display(overview)

In [ ]:
MISSING_LABEL = "(нет данных)"
MIN_GROUP_SIZE = 30


def salary_stats_by_category(data, column, min_n=MIN_GROUP_SIZE):
    prepared = data.copy()
    prepared[column] = prepared[column].fillna(MISSING_LABEL)

    stats = (
        prepared.groupby(column)
        .agg(
            n=("salary_min", "size"),
            min_median=("salary_min", "median"),
            max_median=("salary_max", "median"),
            min_q1=("salary_min", lambda values: values.quantile(0.25)),
            min_q3=("salary_min", lambda values: values.quantile(0.75)),
        )
        .sort_values("min_median", ascending=False)
    )
    stats["median_vs_all"] = (stats["min_median"] / data["salary_min"].median()).round(2)

    result = stats[stats["n"] >= min_n].copy()
    salary_columns = ["min_median", "max_median", "min_q1", "min_q3"]
    result[salary_columns] = result[salary_columns].round(0)

    return result


def plot_salary_by_category(data, column, min_n=MIN_GROUP_SIZE):
    prepared = data.copy()
    prepared[column] = prepared[column].fillna(MISSING_LABEL)

    counts = prepared[column].value_counts()
    prepared = prepared[prepared[column].isin(counts[counts >= min_n].index)]
    order = prepared.groupby(column)["salary_min"].median().sort_values(ascending=False).index

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, salary_col in zip(axes, ["salary_min", "salary_max"]):
        sns.boxplot(data=prepared, x=column, y=salary_col, order=order, ax=ax, showfliers=False)
        ax.axhline(data[salary_col].median(), color="red", linestyle="--", linewidth=1)
        ax.set_title(f"{salary_col} по признаку {column}")
        ax.set_xlabel("")
        ax.set_ylabel("USD в год")
        ax.tick_params(axis="x", rotation=40)

    plt.tight_layout()
    plt.show()

У признаков с высокой кардинальностью (**title** — 8044, **location** — 4372, **company** — 3970, **department** — 2213, **city** — 800) почти каждая категория встречается один раз, поэтому напрямую по группам их не сравниваю — нужна агрегация (роль из title, отрасль из department).

Для сравнения берём признаки с разумным числом категорий: **ats**, **employment_type**, **workplace_type**, **seniority**, **salary_source**, **country_code**.

**ats** — платформа вакансии. Косвенно показывает тип компаний: через Ashby публикуются в основном стартапы, через Workday — крупный корпоративный сектор.

In [ ]:
display(salary_stats_by_category(df, "ats"))
plot_salary_by_category(df, "ats")

**employment_type** — тип занятости. Ожидаемо ниже всего part-time и temporary, выше всего full-time.

In [ ]:
display(salary_stats_by_category(df, "employment_type"))
plot_salary_by_category(df, "employment_type")

**workplace_type** — формат работы. Проверяем расхожее мнение, что на remote платят больше, чем onsite.

In [ ]:
display(salary_stats_by_category(df, "workplace_type"))
plot_salary_by_category(df, "workplace_type")

**seniority** — уровень позиции. Признак заполнен плохо (около 630 значений), поэтому здесь порог снижен до n ≥ 20, чтобы попала группа executive (24 наблюдения). Internship (15) всё равно отсеивается.

In [ ]:
display(salary_stats_by_category(df, "seniority", min_n=20))
plot_salary_by_category(df, "seniority", min_n=20)

**salary_source** — откуда взята зарплата: из поля ATS или из текста вакансии. Разные источники по-разному указывают период и валюту, поэтому часть различий — следствие сбора данных, а не рынка.

In [ ]:
display(salary_stats_by_category(df, "salary_source"))
plot_salary_by_category(df, "salary_source")

**country_code** — страна вакансии. В расчёт попадают только страны с n ≥ 30.

In [ ]:
display(salary_stats_by_category(df, "country_code"))
plot_salary_by_category(df, "country_code")

**language** и **region** отдельно не разбираем: у language с n ≥ 30 остаётся только `en` (3389), остальные группы по 1–35 наблюдений, а region — 244 значения с очень неравномерным заполнением.

### Итоги по категориальным признакам

- **ats** — самый заметный признак: медиана salary_min от 44 811 (Recruitee) до 130 000 (Ashby), почти в 3 раза. Для salary_max картина та же: 61 196 против 170 500.
- **workplace_type** — remote (110 000) > hybrid (100 000) > onsite (66 667); «нет данных» (76 700) между onsite и hybrid.
- **employment_type** — full-time 84 892, part-time 37 440 (в 2.3 раза меньше), temporary 47 840; contract (72 000) ниже full-time. Группы contract/temporary/internship небольшие (124 / 104 / 53).
- **seniority** — порядок ожидаемый: executive 100 000 > senior 83 200 > mid 50 711 > entry 44 860. Признак сильный, но 93% пропусков, так что его нужно восстанавливать из title.
- **country_code** — среди стран с n ≥ 30 выше всех США (90 000), ниже всех Франция и Италия (~36 000). При этом группа «нет данных» (110 000) выше всех стран: отсутствие страны связано с более высокими зарплатами.
- **salary_source** — ats 100 000 против description 80 000, и доля выбросов в ats-выборке выше (4.4% против 2.8%).

Пропуски образуют большие отдельные категории (workplace_type 56%, seniority 93%, language 62%), и это не случайная подвыборка — у них своя медиана и свои платформы-источники. Значимость различий проверяем отдельно, в `hypothesis_testing.ipynb`.

## Числовые выбросы

Числовых признаков четыре: **salary_min**, **salary_max**, **salary_min_annual**, **salary_max_annual**. Разница в том, что `annual`-колонки приведены к году, но **без пересчёта валюты**, а `salary_min` / `salary_max` — и к году, и к USD. Поэтому основными берём последние, а annual показываем рядом для сравнения.

Ищем выбросы двумя способами: по **межквартильному размаху (Тьюки)** — границы `Q1 − 1.5·IQR` и `Q3 + 1.5·IQR`, и по **z-оценке** (|z| > 3). Первый устойчив к асимметрии, второй даёт более строгий фильтр.

Удалять такие строки сразу нельзя — часть из них настоящие высокие зарплаты, поэтому сначала смотрим квантили, потом разбираем причины.

In [ ]:
salary_cols = ["salary_min", "salary_max", "salary_min_annual", "salary_max_annual"]

quantile_levels = [0.0, 0.001, 0.005, 0.01, 0.05, 0.5, 0.95, 0.99, 0.995, 0.999, 1.0]

quantile_table = df[salary_cols].quantile(quantile_levels).round(0)
quantile_table.loc["mean"] = df[salary_cols].mean().round(0)
quantile_table.loc["std"] = df[salary_cols].std().round(0)
quantile_table.loc["skew"] = df[salary_cols].skew().round(2)
quantile_table.loc["mean / median"] = (df[salary_cols].mean() / df[salary_cols].median()).round(2)

display(quantile_table)

По квантилям проблема видна сразу: медиана salary_min — 85 000, максимум — 1 168 539 (в 13.7 раза больше), у salary_max — 112 380 против 1 380 000. Асимметрия 3.4–3.8, среднее к медиане 1.18.

Ещё любопытно расхождение минимумов: у `salary_min` минимум 2 051 (польская вакансия в злотых), а у `salary_min_annual` — 8 000, потому что annual-колонки не пересчитаны в валюту.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, salary_col in zip(axes[0], ["salary_min", "salary_max"]):
    sns.histplot(df[salary_col].dropna(), bins=50, log_scale=True, ax=ax)
    ax.set_title(f"Распределение {salary_col} (логарифмическая шкала)")
    ax.set_xlabel("USD в год")

for ax, salary_col in zip(axes[1], ["salary_min", "salary_max"]):
    sns.boxplot(x=df[salary_col].dropna(), ax=ax)
    ax.set_title(f"Ящик с усами: {salary_col} (выбросы показаны точками)")
    ax.set_xlabel("USD в год")

plt.tight_layout()
plt.show()

sns.scatterplot(data=df, x="salary_min", y="salary_max", alpha=0.3, s=12)
plt.title("Соотношение salary_min и salary_max")
plt.xlabel("salary_min, USD в год")
plt.ylabel("salary_max, USD в год")
plt.tight_layout()
plt.show()

print("Корреляция salary_min и salary_max:")
display(df[["salary_min", "salary_max"]].corr(method="pearson").round(3))
display(df[["salary_min", "salary_max"]].corr(method="spearman").round(3))

Гистограмма подтверждает длинный правый хвост, а на boxplot'ах видно, что выбросы только сверху: нижняя граница Тьюки уходит в минус, так что формально «нижних» выбросов нет.

salary_min и salary_max движутся вместе — корреляция 0.944 (Pearson) и 0.958 (Spearman), так что анализировать их можно одинаково.

In [ ]:
def iqr_bounds(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


outlier_report = []

for salary_col in ["salary_min", "salary_max"]:
    values = df[salary_col]
    low, high = iqr_bounds(values.dropna())
    z_scores = (values - values.mean()) / values.std()

    df[f"outlier_iqr_{salary_col}"] = (values < low) | (values > high)
    df[f"outlier_z_{salary_col}"] = z_scores.abs() > 3

    outlier_report.append({
        "признак": salary_col,
        "нижняя граница (IQR)": round(low),
        "верхняя граница (IQR)": round(high),
        "ниже нижней границы": int((values < low).sum()),
        "выше верхней границы": int((values > high).sum()),
        "|z| > 3": int((z_scores.abs() > 3).sum()),
        "|z| > 4": int((z_scores.abs() > 4).sum()),
    })

df["outlier_any"] = df["outlier_iqr_salary_min"] | df["outlier_iqr_salary_max"]

display(pd.DataFrame(outlier_report).set_index("признак"))
print("Строк, где зарплата выходит за границы IQR хотя бы по одному признаку:", int(df["outlier_any"].sum()))

По Тьюки выбросов сверху ~250 для salary_min и ~206 для salary_max (2.7% и 2.2% наблюдений), по z-оценке строже — 90 и 85. Нижних выбросов нет ни у одного признака.

Теперь главный вопрос: это реальные зарплаты или ошибки пересчёта? Посмотрим сами строки вместе с исходными значениями и текстом.

In [ ]:
outlier_rows = df[df["outlier_iqr_salary_min"]].copy()
outlier_rows["raw_min"] = df_original.loc[outlier_rows.index, "salary_min"]
outlier_rows["raw_max"] = df_original.loc[outlier_rows.index, "salary_max"]
outlier_rows["raw_period"] = df_original.loc[outlier_rows.index, "salary_period"]
outlier_rows["raw_currency"] = df_original.loc[outlier_rows.index, "salary_currency"]

columns_to_show = [
    "title",
    "salary_text",
    "raw_min",
    "raw_max",
    "raw_period",
    "raw_currency",
    "salary_min",
    "salary_max",
    "salary_source",
    "country_code",
]

display(outlier_rows.sort_values("salary_min", ascending=False)[columns_to_show].head(15))

Первые строки — как раз ошибки нормализации: в **salary_text** указана годовая сумма (`$85,000–$115,000`, `€500-550`), но период помечен как `month` или `hour`, и после умножения получается 1.0–1.4 млн. Проверим, насколько это системно.

In [ ]:
df_work = df.copy()
df_work["raw_period"] = df_original["salary_period"]
df_work["outlier_label"] = df_work["outlier_any"].map({True: "выброс", False: "остальные"})

period_share = (
    pd.crosstab(df_work["raw_period"], df_work["outlier_label"], normalize="index")
    .mul(100)
    .round(1)
)
period_share["всего вакансий"] = df_work["raw_period"].value_counts()

source_share = (
    pd.crosstab(df_work["salary_source"], df_work["outlier_label"], normalize="index")
    .mul(100)
    .round(1)
)
source_share["всего вакансий"] = df_work["salary_source"].value_counts()

display(period_share)
display(source_share)

print("Выбросов по salary_min:", int(df["outlier_iqr_salary_min"].sum()))
print("Выбросов по salary_max:", int(df["outlier_iqr_salary_max"].sum()))
print("Выбросов хотя бы по одному признаку:", int(df["outlier_any"].sum()))
print(
    "Из них с исходным периодом, отличным от года:",
    int((df["outlier_any"] & (df_original["salary_period"] != "year")).sum()),
)

Среди вакансий с периодом `month` в выбросы попадает 7.7% (21 из 274) — почти вдвое больше, чем при годовом периоде (4.0%). А у `hour` доля минимальна (0.9%): почасовые ставки после умножения на 2080 обычно попадают в нормальный диапазон.

Так что выбросы делятся на три группы:

1. **Ошибки периода** (~43 строки из 307) — в тексте годовая сумма, а период помечен как `month` или `hour`. Это ошибка данных, а не рынка.
2. **Реально высокие зарплаты** — 99.5-й квантиль salary_min 380 000, 99.9-й — 720 000 (senior и руководящие позиции в США). Естественный хвост, убирать его нельзя.
3. **Низкий хвост** — почасовые и part-time вакансии (`$5.00/hour`, `$700/month`) и зарплаты в злотых (~2 051 USD/год). По IQR это не выбросы, но в шкале USD/год они сильно тянут вниз.

Поэтому «просто отрезать всё выше 250 000» — плохое решение: вместе с ошибками уйдут реальные высокие зарплаты. Лучше пометить строки флагами и завести сглаженные колонки.

In [ ]:
winsor_bounds = {}

for salary_col in ["salary_min", "salary_max"]:
    low, high = df[salary_col].quantile([0.01, 0.99])
    winsor_bounds[salary_col] = (round(low), round(high))
    df[f"{salary_col}_winsor"] = df[salary_col].clip(low, high)

winsor_compare = pd.DataFrame(
    {
        "было: среднее": df[["salary_min", "salary_max"]].mean().round(0).to_numpy(),
        "стало: среднее": df[["salary_min_winsor", "salary_max_winsor"]].mean().round(0).to_numpy(),
        "было: стандартное отклонение": df[["salary_min", "salary_max"]].std().round(0).to_numpy(),
        "стало: стандартное отклонение": df[["salary_min_winsor", "salary_max_winsor"]].std().round(0).to_numpy(),
        "изменено значений": [
            int((df["salary_min"] != df["salary_min_winsor"]).sum()),
            int((df["salary_max"] != df["salary_max_winsor"]).sum()),
        ],
    },
    index=["salary_min", "salary_max"],
)

print("Границы winsorization (1% и 99% квантили):", winsor_bounds)
display(winsor_compare)

### Что сделано с выбросами

Строки не удаляются — иначе потеряем реальные высокие зарплаты. Вместо этого добавлены:

- флаги **`outlier_iqr_salary_min`**, **`outlier_iqr_salary_max`**, **`outlier_z_salary_min`**, **`outlier_z_salary_max`** и общий **`outlier_any`**;
- сглаженные колонки **`salary_min_winsor`** и **`salary_max_winsor`** (обрезка по 1% и 99% квантилям): среднее почти не меняется, а стандартное отклонение падает с 69 470 до 58 624 и с 93 543 до 80 093.

Как использовать: для описательных статистик — winsorized-колонки, для моделей — исходные значения плюс флаг `outlier_any`, а ≈43 строки с подозрительным `salary_period` стоит перепроверить отдельно.

## Общий вывод

1. **Категориальные признаки.** Сильнее всего с зарплатой связаны **ats** (медиана от 44 811 до 130 000), **seniority** (entry 44 860 → executive 100 000, но 93% пропусков), **workplace_type** (remote 110 000 против onsite 66 667), **employment_type** (part-time 37 440 против full-time 84 892) и **country_code** (лидируют США, 90 000). Признаки с высокой кардинальностью (title, company, department, city) сначала нужно агрегировать.

2. **Числовые выбросы.** Распределение зарплат сильно смещено вправо (skew ≈ 3.8), выбросы только сверху: ~250 строк по salary_min и ~206 по salary_max по Тьюки, 90 и 85 по z-оценке. Большинство — реальные высокие зарплаты, но ≈43 строки появились из-за неверно указанного `salary_period`. Обработано флагами и winsorization, без удаления строк.